# Lecture: Relationships Between Categorical Features

A frequency table or bar chart describes the distribution of one categorical feature. Many questions ask whether the distribution of an outcome differs across groups. In this lecture, we will use Titanic passenger records and a simulated A/B test to compare two categorical features.


## Learning goals

By the end of this lecture, you should be able to:

- identify a target feature and a comparison feature;
- create a new categorical feature from a quantitative count;
- create a cross-tabulation of two categorical features;
- distinguish counts, joint percentages, and conditional percentages;
- explain why percentages provide a fairer comparison when groups have different sizes;
- visualize conditional percentages with a bar chart; and
- state a conclusion and limitation supported by the analysis.


## Import the libraries

We use Pandas to read and organize the data and Matplotlib to label and refine the figures.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# Part 1: Survival of Titanic Passengers

RMS *Titanic* sank in the North Atlantic in April 1912. The passenger dataset contains one row for each of 891 passengers. It includes personal and travel information such as passenger name, age, sex, passenger class, family members aboard, port of departure, fare, and survival.

This is a passenger-only dataset. It does not include the ship's crew.


## Data dictionary

| Feature | Description |
| --- | --- |
| `PassengerId` | Unique passenger identifier |
| `Survived` | Survival outcome: 0 = did not survive, 1 = survived |
| `Pclass` | Passenger class: 1 = first, 2 = second, 3 = third |
| `Name` | Passenger name |
| `Sex` | Sex recorded as male or female |
| `Age` | Age in years |
| `SibSp` | Number of siblings or spouses aboard |
| `Parch` | Number of parents or children aboard |
| `Ticket` | Ticket number |
| `Fare` | Passenger fare |
| `Cabin` | Cabin number, when recorded |
| `Embarked` | Port of departure: C = Cherbourg, Q = Queenstown, S = Southampton |

Historical categories and missing values reflect the surviving records and the way this commonly used dataset was prepared.


## Discussion: predict the target feature

1. Which feature should be the target?
2. What does the target feature represent?


## The reproducible analysis workflow

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Evidence:** What should we calculate and display to answer the question?
4. **Check:** Does the result make sense in context? Do a few original records support the meaning of the calculation?
5. **Conclusion:** What claim is supported?
6. **Limitation:** What should we avoid concluding?


# Research question 1: Who was more likely to survive?


## Step 1 — Question

> **Who was more likely to survive: male or female passengers aboard the Titanic?**

The target feature is `Survived`. The comparison feature is `Sex`.

### Discussion: predict the result

1. Which group do you predict had the higher survival percentage?
2. What historical assumptions or prior knowledge influenced your prediction?


Your prediction:


## Step 2 — Data

Load the local passenger file and inspect the DataFrame. These checks use the same techniques introduced earlier in the course.


In [ ]:
titanic_df = pd.read_csv("data/titanic_passengers.csv")
titanic_df.head()


In [ ]:
titanic_df.tail()


In [ ]:
titanic_df.info()


In [ ]:
titanic_df.isnull().sum()


### Discussion: check the data

1. What does each row represent?
2. Which features are needed for the first research question?
3. Are either of those features missing values?
4. Which other features have missing values?


## Step 3 — Evidence

Begin by examining the distribution of each categorical feature. `.value_counts()` reports the frequency of every category. Adding `normalize=True` divides each frequency by the total, and multiplying by 100 converts the proportions to percentages.


In [ ]:
print(titanic_df["Sex"].value_counts())
print(titanic_df["Sex"].value_counts(normalize=True).mul(100).round(1))


In [ ]:
print(titanic_df["Survived"].value_counts())
print(titanic_df["Survived"].value_counts(normalize=True).mul(100).round(1))


### Joint distribution: both features together

A **cross-tabulation**, or cross-tab, displays every combination of two categorical features. The joint distribution describes how observations are distributed across those combinations: female and survived, female and did not survive, male and survived, and male and did not survive.

A count cross-tab shows the joint distribution as frequencies. To express it as **joint percentages**, divide every cell by the total number of passengers: `normalize="all"`. All cells together then add to 100%, rather than each row adding to 100%.


In [ ]:
survival_counts = pd.crosstab(
    titanic_df["Sex"],
    titanic_df["Survived"]
)

survival_counts


Switch the feature order: the first argument supplies rows and the second supplies columns. Each count stays attached to the same combination; only the table orientation changes. Row-normalized percentages would condition on a different feature after this swap, so the same claim does not apply to conditional percentages.


In [ ]:
pd.crosstab(titanic_df["Survived"], titanic_df["Sex"])


A joint percentage answers “What percent of **all passengers** were both female and survivors?” Display the joint percentages and compare their denominator with the count table.


In [ ]:
pd.crosstab(titanic_df["Sex"], titanic_df["Survived"], normalize="all").mul(100).round(1)


### Conditional distribution: the outcome within a group

A **conditional distribution** describes one feature after restricting attention to a category of the other. Here we condition on sex and examine survival within each sex. “What percent of **female passengers** survived?” uses only female passengers as the denominator.

Counts do not create a fair comparison when the groups contain different numbers of passengers. The question asks for the survival percentage **within each sex**, so each sex must have its own denominator.

`normalize="index"` divides every cell by its row total. Multiplying by 100 makes each row add to 100%. Conditioning on survival instead (using column totals) would answer a different question: the sex distribution within each survival outcome.

| Representation | Denominator | What adds to 100%? |
| --- | --- | --- |
| Joint counts | None; these are numbers of passengers | Not applicable |
| Joint percentages (`normalize="all"`) | All passengers | The entire table |
| Conditional percentages (`normalize="index"`) | Passengers in that row's sex group | Each row |

For example, 233 of 891 passengers were female survivors (26.2% jointly), while 233 of 314 female passengers survived (74.2% conditionally). The numerator is the same; the denominator and question differ.


In [ ]:
survival_percent = (
    pd.crosstab(
        titanic_df["Sex"],
        titanic_df["Survived"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

survival_percent


### Discussion: counts or percentages?

1. Are the male and female groups the same size?
2. Why would comparing only the number of survivors be unfair?
3. In the percentage table, what population is used as the denominator for each row?


### Compare count and conditional-percentage bar charts

First plot the joint counts. Each full bar shows the number of passengers in that sex group, and its sections show the numbers who did and did not survive. Keep the category order and colors the same in both charts so the change in scale is easy to see.


In [ ]:
survival_counts.plot(kind="bar", stacked=True, color=["#B8A3B5", "#65988A"])
plt.title("Titanic Survival by Sex: Joint Counts")
plt.xlabel("Sex")
plt.ylabel("Number of passengers")
plt.xticks(rotation=0)
plt.legend(labels=['Did not survive', 'Survived'], loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


Now plot the conditional percentages. Each complete bar reaches 100%. Use `plt.legend(labels=[...])` to relabel outcomes in plotted column order (0, then 1), without renaming the table columns. Muted mauve represents nonsurvivors and sage green represents survivors.

If a legend covers the bars, move it outside: `loc="upper left"` selects the legend corner and `bbox_to_anchor=(1.02, 1)` places that corner just beyond the right edge of the axes. `plt.tight_layout()` leaves room for labels and the legend.


In [ ]:
survival_percent.plot(kind="bar", stacked=True, color=["#B8A3B5", "#65988A"])
plt.title("Titanic Survival by Sex: Conditional Percentages")
plt.xlabel("Sex")
plt.ylabel("Percent within each sex")
plt.xticks(rotation=0)
plt.ylim(0, 100)
plt.legend(labels=['Did not survive', 'Survived'], loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


### A simpler view: show only the percentage who survived

Select column `1` to show survival alone. With two complete outcome categories, the nonsurvival percentage is 100 minus the survival percentage. This chart communicates the same conditional distribution with fewer marks; it does not show the group sizes.


In [ ]:
survival_percent[1].plot(kind="bar", color="#65988A", legend=True)
plt.title("Titanic Survival Percentage by Sex")
plt.xlabel("Sex")
plt.ylabel("Percent who survived within each sex")
plt.xticks(rotation=0)
plt.ylim(0, 100)
plt.legend(labels=['Survived'], loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


### Discussion: interpret the evidence

1. Why is the male bar taller in the count chart, while both bars reach 100% in the conditional-percentage chart?
2. What does the sage-green section measure in each chart? Which chart directly answers our question about survival likelihood within each sex?
3. Approximately what percentage of each group survived? Does this support your prediction?
4. Which is easier to interpret: the stacked conditional-percentage chart or the survivor-only chart? How can you recover the percentage who did not survive?
5. If we plotted joint percentages instead, would both full bars reach 100%? Explain using the denominator.


## Step 4 — Check

**Does the survival percentage use all passengers of that sex as its denominator, or only survivors? Looking at the count table, does the larger or smaller share of survivors in each group agree with the table and figure? What survival percentages would make you suspicious because they would not make sense?**


## Step 5 — Conclusion

In this passenger dataset, female passengers were more likely to survive than male passengers. The difference is large: roughly three-quarters of female passengers survived, compared with fewer than one-fifth of male passengers.


## Step 6 — Limitation

This is an observational historical dataset, not a randomized experiment. Sex was related to survival, but it was also related to decisions about evacuation and may have been related to age, passenger class, cabin location, and access to lifeboats. The comparison does not establish that sex alone caused the difference.


# Research question 2: Traveling with parents or children


## Step 1 — Question

> **Were passengers traveling with parents or children more likely to survive than passengers traveling without parents or children?**

`Parch` is a count, but the question asks about two groups. We will create a categorical comparison feature before calculating survival percentages.

### Discussion: predict the result

1. Which group do you predict had the higher survival percentage?
2. Why might traveling with parents or children be related to survival?


Your prediction:


## Step 2 — Data

Use `Parch` and `Survived`. We already inspected missing values. Look at the frequency of each number of parents or children aboard.


In [ ]:
titanic_df["Parch"].value_counts().sort_index()


Some `Parch` categories have very few observations, so their percentages can change greatly with just one passenger. Collapsing categories makes the comparison easier to interpret, but can hide differences. Two reasonable choices are:

- **Two groups:** no parents/children versus any parents/children.
- **Three groups:** none, one, or two or more.

Begin with two groups, then compare with three. Choose groups for their meaning and support in the data, not to produce a preferred result.


## Step 3 — Evidence

Create `with_parents_children`: 0 means no parent or child recorded aboard; 1 means at least one. The Boolean comparison converts to integers using `.astype(int)`. Keep the original `Parch` counts.


In [ ]:
titanic_df["with_parents_children"] = (titanic_df["Parch"] > 0).astype(int)


Check the coding across every original category using a cross-tab, rather than previewing a few rows. Do all zero counts map to 0 and all positive counts map to 1?


In [ ]:
pd.crosstab(titanic_df["Parch"], titanic_df["with_parents_children"])


### Joint distribution: counts


In [ ]:
family_survival_counts = pd.crosstab(titanic_df["with_parents_children"], titanic_df["Survived"])
family_survival_counts


**Is this conclusion correct?**

> Because 233 people traveling without parents and children survived, compared with 109 traveling with parents and children, people traveling without parents and children were more likely to survive.

What information is missing from that reasoning?


In [ ]:
family_survival_counts.plot(kind="bar", stacked=True, color=["#B8A3B5", "#65988A"])
plt.title("Survival and Parents/Children: Counts")
plt.xlabel("Parents or children aboard")
plt.ylabel("Number of passengers")
plt.xticks([0, 1], ['None', 'Any'], rotation=0)
plt.legend(labels=['Did not survive', 'Survived'], loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


### Conditional distribution: percentages within each group

Use the number of passengers in each travel group as its denominator.


In [ ]:
family_survival_percent = pd.crosstab(
    titanic_df["with_parents_children"], titanic_df["Survived"], normalize="index"
).mul(100).round(1)
family_survival_percent


In [ ]:
family_survival_percent.plot(kind="bar", stacked=True, color=["#B8A3B5", "#65988A"])
plt.title("Survival and Parents/Children: Conditional Percentages")
plt.xlabel("Parents or children aboard")
plt.ylabel("Percent within each travel group")
plt.xticks([0, 1], ['None', 'Any'], rotation=0)
plt.ylim(0, 100)
plt.legend(labels=['Did not survive', 'Survived'], loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


**Which chart is easier to interpret for our research question: raw numbers or conditional percentages? Which group has the higher survival percentage, and how does that compare with your prediction?**

If a legend overlaps the data, how could you move it? Explain the `loc` and `bbox_to_anchor` arguments used above.


### Repeat with three groups: none, one, or two or more

Does combining everyone with any parents/children hide a difference between one and two or more? Record a prediction before calculating.


Your prediction:


In [ ]:
titanic_df["parents_children_group"] = titanic_df["Parch"]
titanic_df.loc[titanic_df["Parch"] >= 2, "parents_children_group"] = 2
pd.crosstab(titanic_df["Parch"], titanic_df["parents_children_group"])


Here 0 means none, 1 means one, and 2 means two or more. Reuse the same calculations and charts.


In [ ]:
family_three_counts = pd.crosstab(titanic_df["parents_children_group"], titanic_df["Survived"])
family_three_counts


In [ ]:
family_three_percent = pd.crosstab(
    titanic_df["parents_children_group"], titanic_df["Survived"], normalize="index"
).mul(100).round(1)
family_three_percent


In [ ]:
family_three_counts.plot(kind="bar", stacked=True, color=["#B8A3B5", "#65988A"])
plt.title("Three Parents/Children Groups: Counts")
plt.xlabel("Parents or children aboard")
plt.ylabel("Number of passengers")
plt.xticks([0, 1, 2], ['None', 'One', 'Two or more'], rotation=0)
plt.legend(labels=['Did not survive', 'Survived'], loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


In [ ]:
family_three_percent.plot(kind="bar", stacked=True, color=["#B8A3B5", "#65988A"])
plt.title("Three Parents/Children Groups: Conditional Percentages")
plt.xlabel("Parents or children aboard")
plt.ylabel("Percent within each travel group")
plt.xticks([0, 1, 2], ['None', 'One', 'Two or more'], rotation=0)
plt.ylim(0, 100)
plt.legend(labels=['Did not survive', 'Survived'], loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


**What additional information does the three-group comparison reveal? Does it suggest that survival always increases as the number of parents/children increases? What detail is still hidden within “two or more”?**


## Step 4 — Check

Review the coding cross-tabs: does each original `Parch` value appear only in its intended group? Do the conditional percentages use all passengers in each group as the denominator? A zero does not establish that a passenger traveled alone: siblings, spouses, or other companions may still have been aboard.

### Checking code is not the same as verifying historical records

Our cross-tabs can confirm how we coded this file, but cannot establish that every historical entry is accurate. Titanic datasets have been reconstructed and revised; we cannot be absolutely certain of every recorded value. Compare these documented versions:

- [Titanic dataset history and titanic3](https://hbiostat.org/data/repo/titanic): the maintainers describe removing duplicate passengers, correcting errors, and filling missing ages.
- [titanic5 update and comparison with titanic3](https://hbiostat.org/data/repo/titanic5): the compiler reports differences especially in age, with 51 missing ages rather than 263 among 1,309 passengers, and documents how ages were derived.

These are documented disagreements and revisions, not proof that a particular `Parch` entry in our file is wrong. Different row totals can also reflect different coverage, such as including crew or using a training subset. Before comparing values, match the same people and feature definitions.

**If two sources disagree, what documentation would you consult? How should uncertainty about the historical records affect the strength of our conclusion?**


## Step 5 — Conclusion

In this dataset, passengers recorded with any parents/children had a higher survival percentage than those with none (51.2% versus 34.4%). Splitting the former group reveals 55.1% for one and 46.3% for two or more. These results describe recorded groups, not an effect caused by traveling with family.


## Step 6 — Limitation

`Parch` records only the number of parents or children aboard. The dataset documentation notes that some children who traveled with a nanny have `Parch` equal to 0. The feature also does not describe every family relationship, who stayed together during the evacuation, or whether a passenger was traveling with other companions. The groups may differ in age, class, and other features related to survival. This relationship should not be interpreted as proof that traveling with parents or children caused survival.


# Part 2: A/B Testing

An **A/B test** randomly assigns participants to one of two versions of a product, message, or experience. The outcome is then compared across the two experimental conditions.

Random assignment is what makes an A/B test different from the Titanic comparisons. When the groups are created randomly and the experiment is conducted well, a difference in the response can provide evidence that the tested version affected the outcome.


## Example: a museum membership page

A science museum wants more website visitors to begin a free membership. It randomly assigns visitors to one of two pages:

- `Standard page`: the existing page with several membership choices;
- `Simplified page`: a shorter page with one prominent free-membership button.

The response feature `Signed_up` records whether each visitor completed the free-membership signup.

This is a **simulated teaching dataset**. The observations were created to make the structure and interpretation of an A/B test easy to see; they are not evidence about a real museum or website.


## Step 1 — Question

> **Did visitors shown the simplified page sign up more often than visitors shown the standard page?**

### Discussion: identify the features and predict

1. Which feature is the experimental condition?
2. Which feature is the response?
3. Which page do you predict will have the higher signup percentage?


Your prediction:


## Step 2 — Data


In [ ]:
signup_df = pd.read_csv("data/museum_signup_ab.csv")
signup_df.head()


In [ ]:
signup_df.info()


In [ ]:
signup_df.isnull().sum()


## Step 3 — Evidence

Inspect the condition and response frequencies, then calculate counts and signup percentages within each page version.


In [ ]:
print(signup_df["Page_version"].value_counts())
print(signup_df["Signed_up"].value_counts())


In [ ]:
signup_counts = pd.crosstab(
    signup_df["Page_version"],
    signup_df["Signed_up"]
)

signup_counts


In [ ]:
signup_percent = (
    pd.crosstab(
        signup_df["Page_version"],
        signup_df["Signed_up"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

signup_percent


### Compare signup frequencies and conditional percentages

Plot only the `Yes` column in each figure: first the number who signed up, then the percentage who signed up within each page version. Omitting `No` simplifies the figures without changing the outcome being compared.


In [ ]:
signup_counts["Yes"].plot(kind="bar", color="#65988A", legend=True)
plt.title("Membership Signups by Page: Counts")
plt.xlabel("Page version")
plt.ylabel("Number who signed up")
plt.xticks(rotation=0)
plt.legend(labels=['Signed up'], loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


In [ ]:
signup_percent["Yes"].plot(kind="bar", color="#65988A", legend=True)
plt.title("Membership Signups by Page: Conditional Percentages")
plt.xlabel("Page version")
plt.ylabel("Percent who signed up within each page")
plt.xticks(rotation=0)
plt.ylim(0, 100)
plt.legend(labels=['Signed up'], loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


### Discussion: interpret the A/B test

1. What percentage of visitors signed up after seeing each page?
2. What is the difference in percentage points?
3. Which figure is easier to interpret? Here both page groups contain 60 visitors; how would unequal group sizes affect the count comparison?
4. Does the result support your prediction?


## Step 4 — Check

**Does the count table show a majority signing up for the simplified page and a minority for the standard page, as the percentages and figure suggest? Are we calculating the percentage who signed up within each page version, or the percentage of signups who saw each page? What would make you suspicious?**


## Step 5 — Conclusion

In this simulated experiment, visitors assigned to the simplified page signed up substantially more often than visitors assigned to the standard page.


## Step 6 — Limitation

Because this dataset was simulated for teaching, the result cannot support a real product decision. In a real A/B test, we would also verify that random assignment and data collection worked as intended and use statistical inference to assess uncertainty before deciding whether to change the website.


## Procedure for comparing categorical features

1. Identify the target or response feature and the comparison feature.
2. Inspect the frequency of each feature.
3. Create a count cross-tab.
4. Identify the denominator implied by the research question.
5. Calculate the joint or conditional percentages required by the question.
6. Choose a figure that makes the comparison clear.
7. Check whether the values, group meanings, and denominators make sense in context.
8. State a conclusion and limitation.


## Charles Joseph Minard and the development of flow maps

Charles Joseph Minard (1781–1870) was a French civil engineer who pioneered the use of flow maps. His best-known visualization, published in 1869, depicts the losses suffered by Napoleon's army during the 1812 campaign in Russia.

Minard combined several features in one figure:

- geographic location and the army's direction of travel;
- the number of troops, represented by the width of the band;
- the advance toward Moscow in tan and the retreat in black; and
- dates and temperatures during the retreat.

The narrowing bands make the army's losses visible without requiring the reader to inspect every printed number.


<img src="images/minard_napoleon_1869.png" alt="Charles Joseph Minard's 1869 flow map of Napoleon's 1812 campaign in Russia" width="920">

*Minard's 1869 flow map of Napoleon's 1812 Russian campaign. [Original image from Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Minard.png), public domain.*


### Discussion

1. What question does Minard's visualization help answer?
2. Which feature is represented by the width of the band?
3. What conclusion becomes visible as the band narrows?
4. What information or limitations would you want before using the figure as historical evidence?


## Key takeaways

- The target or response feature is the outcome we want to understand.
- A cross-tab describes the joint distribution of two categorical features.
- Counts describe how many observations appear in each combination of categories.
- Conditional percentages describe the response within each comparison group.
- `normalize="index"` makes every row add to 100%.
- A calculated categorical feature can turn a quantitative count into groups that match a research question.
- Random assignment distinguishes an experiment from an observational comparison.
- An observed relationship should be reported with a limitation and should not automatically be interpreted as causal.
